Testing the same code for premier league

In [1]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
def get_PL_matches(season: int) -> pd.DataFrame :
    url = f"https://raw.githubusercontent.com/openfootball/football.json/master/{season}-{season%100 + 1}/en.1.json"
    data = requests.get(url).json()

    rows = []
    for match in data["matches"]:
        rows.append({
            "matchday": match["round"],
            "date": match["date"],
            "home": match["team1"],
            "away": match["team2"],
            "score_home": match["score"]["ft"][0],
            "score_away": match["score"]["ft"][1]
        })
    matches = pd.DataFrame(rows)
    return matches

In [56]:
get_PL_matches(2024)

,matchday,date,home,away,score_home,score_away
0,Matchday 1,2024-08-16,Manchester United FC,Fulham FC,1,0
1,Matchday 1,2024-08-17,Ipswich Town FC,Liverpool FC,0,2
2,Matchday 1,2024-08-17,Arsenal FC,Wolverhampton Wanderers FC,2,0
3,Matchday 1,2024-08-17,Everton FC,Brighton & Hove Albion FC,0,3
4,Matchday 1,2024-08-17,Newcastle United FC,Southampton FC,1,0
...,...,...,...,...,...,...
375,Matchday 38,2025-05-25,Newcastle United FC,Everton FC,0,1
376,Matchday 38,2025-05-25,Nottingham Forest FC,Chelsea FC,0,1
377,Matchday 38,2025-05-25,Southampton FC,Arsenal FC,1,2
378,Matchday 38,2025-05-25,Tottenham Hotspur FC,Brighton & Hove Albion FC,1,4


In [57]:
def create_PL_table(season: int) -> pd.DataFrame:

    season_matches = get_PL_matches(season)

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            # teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            # teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            # "home_wins" : statistics["home_wins"],
            # "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [90]:
def prep_data(start_date : int, end_date : int): #end date should be 2024
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = create_PL_table(year)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy().set_index("team_name")
        curr_table = season_tables[f"{i +1}"].copy().set_index("team_name")
        relegated = prev_table.tail(3)
        promoted_stats = relegated.mean().to_dict()
        
        for team, row in curr_table.iterrows():
            if team in prev_table.index:
                features = prev_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
            else:
                features = {k : promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
            feature_rows.append(features)
            target_rows.append(row["position"])
    X_train = pd.DataFrame(feature_rows)
    y_train = pd.Series(target_rows)

    return X_train, y_train

X_train, y_train = prep_data(2018, 2023)

    # last_feature_rows = []
    # last_table = season_tables[f"{end_date}"].copy().set_index("team_name")
    # last_relegated = last_table.tail(2)
    # last_promoted_stats = last_relegated.mean().to_dict()
    # last_teams = last_table.index.tolist()

    # last_promoted_teams = ["FC Köln", "Hamburger SV"]
    # last_relegated_teams = last_relegated.index.tolist()

    # for team in last_teams:
    #     if team in last_relegated_teams:
    #         continue
    #     features = last_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
    #     last_feature_rows.append((team, features))
    # for team in last_promoted_teams:
    #     if team not in last_teams:
    #         feats = {k : last_promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
    #         last_feature_rows.append((team, feats))
    # latest_features_df = pd.DataFrame([feats for _, feats in last_feature_rows],
    #                                   index=[t for t, _ in last_feature_rows])
    # return X_train, y_train, latest_features_df

In [91]:
X_train

,points,win,draw,loss,goals_for,goals_against,goal_diff
0,97.0,30.000000,7.0,1.000000,89.0,22.0,67.0
1,98.0,32.000000,2.0,4.000000,95.0,23.0,72.0
2,66.0,19.000000,9.0,10.000000,65.0,54.0,11.0
3,72.0,21.000000,9.0,8.000000,63.0,39.0,24.0
4,52.0,15.000000,7.0,16.000000,51.0,48.0,3.0
...,...,...,...,...,...,...,...
95,59.0,15.000000,14.0,9.000000,58.0,46.0,12.0
96,38.0,9.000000,11.0,18.000000,38.0,68.0,-30.0
97,30.0,7.333333,8.0,22.666667,45.0,73.0,-28.0
98,30.0,7.333333,8.0,22.666667,45.0,73.0,-28.0


Now, we start the random forest using Sklearn.

In [72]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])

    model.fit(X, y)
    return model

In [74]:
X, y, df = prep_data(2015, 2024)

In [75]:
df

,points,win,draw,loss,goals_for,goals_against,goal_diff
Liverpool FC,84.0,25.0,9.0,4.0,86.0,41.0,45.0
Arsenal FC,74.0,20.0,14.0,4.0,69.0,34.0,35.0
Manchester City FC,71.0,21.0,8.0,9.0,72.0,44.0,28.0
Chelsea FC,69.0,20.0,9.0,9.0,64.0,43.0,21.0
Newcastle United FC,66.0,20.0,6.0,12.0,68.0,47.0,21.0
Aston Villa FC,66.0,19.0,9.0,10.0,58.0,51.0,7.0
Nottingham Forest FC,65.0,19.0,8.0,11.0,58.0,46.0,12.0
Brighton & Hove Albion FC,61.0,16.0,13.0,9.0,66.0,59.0,7.0
AFC Bournemouth,56.0,15.0,11.0,12.0,58.0,46.0,12.0
Brentford FC,56.0,16.0,8.0,14.0,66.0,57.0,9.0


In [76]:
model = train_model(X, y)

In [77]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_
exp_positions = prob.dot(classes)
prediction_df = pd.DataFrame({
        "team": df.index,
        "expected_position": exp_positions
        })

In [78]:
prediction_df.sort_values(["expected_position"], ascending=True)
prediction_df["position"] = prediction_df.index +1

In [79]:
prediction_df

,team,expected_position,position
0,Liverpool FC,2.330000,1
1,Arsenal FC,5.600000,2
2,Manchester City FC,6.808333,3
3,Chelsea FC,7.188333,4
4,Newcastle United FC,10.533333,5
5,Aston Villa FC,9.483333,6
6,Nottingham Forest FC,10.026667,7
7,Brighton & Hove Albion FC,9.783333,8
8,AFC Bournemouth,10.476667,9
9,Brentford FC,11.380000,10
